# 19 — SPA (Single Photon Amplifier) Optimization

Consolidates legacy experiments 55, 56, 57, and 58.

1. **SPA flux optimization** — sweep DC flux to find optimal SPA working point (Exp 55)
2. **SPA pump frequency optimization** — sweep pump frequency (Exp 56)
3. **SPA readout discrimination with SPA** — measure readout fidelity with SPA (Exp 57)
4. **SPA IQ blob** — 3-state IQ scatter with SPA active (Exp 58)

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.compat.notebook import (
    IQBlob,
    ReadoutGEDiscrimination,
    SPAFluxOptimization,
    SPAPumpFrequencyOptimization,
    fit_quality_gate,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="19_spa_optimization",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

readout_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if readout_checkpoint is not None:
    print(f"Prior stage 16 status: {readout_checkpoint['status']}")

## 2. SPA Defaults

In [ ]:
APPLY_SPA_CALIBRATION = True

# --- SPA Flux Optimization (Exp 55) ---
SPA_FLUX_N_AVG = 10000

# --- SPA Pump Frequency (Exp 56) ---
SPA_PUMP_N_AVG = 10000

# --- SPA Discrimination (Exp 57) ---
SPA_DISC_N_AVG = 50000

# --- SPA IQ Blob (Exp 58) ---
SPA_IQ_BLOB_N_AVG = 10000

print("SPA optimization settings:")
print(f"  apply SPA calibration: {APPLY_SPA_CALIBRATION}")
print(f"  flux opt n_avg: {SPA_FLUX_N_AVG}")
print(f"  pump freq n_avg: {SPA_PUMP_N_AVG}")
print(f"  discrimination n_avg: {SPA_DISC_N_AVG}")
print(f"  IQ blob n_avg: {SPA_IQ_BLOB_N_AVG}")

## 3. SPA Flux Optimization — Exp 55

Sweep DC flux to find the optimal SPA working point.

In [ ]:
spa_flux = SPAFluxOptimization(session)
spa_flux_result = spa_flux.run(n_avg=SPA_FLUX_N_AVG)
spa_flux_analysis = spa_flux.analyze(spa_flux_result, update_calibration=True)
spa_flux.plot(spa_flux_analysis)

spa_flux_fit_ok = fit_quality_gate(spa_flux_analysis, r_squared_min=0.80)

if spa_flux_fit_ok:
    spa_flux_patch, _, spa_flux_apply = preview_or_apply_patch_ops(
        session,
        reason="SPA flux optimization",
        proposed_patch_ops=spa_flux_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_SPA_CALIBRATION,
    )
    if spa_flux_apply is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
else:
    print("WARNING: SPA flux fit quality gate FAILED — skipping apply.")

print(f"SPA flux optimization complete.")

## 4. SPA Pump Frequency Optimization — Exp 56

Sweep pump frequency to maximize SPA gain.

In [ ]:
spa_pump = SPAPumpFrequencyOptimization(session)
spa_pump_result = spa_pump.run(n_avg=SPA_PUMP_N_AVG)
spa_pump_analysis = spa_pump.analyze(spa_pump_result, update_calibration=True)
spa_pump.plot(spa_pump_analysis)

spa_pump_fit_ok = fit_quality_gate(spa_pump_analysis, r_squared_min=0.80)

if spa_pump_fit_ok:
    spa_pump_patch, _, spa_pump_apply = preview_or_apply_patch_ops(
        session,
        reason="SPA pump frequency optimization",
        proposed_patch_ops=spa_pump_analysis.metadata.get("proposed_patch_ops", []),
        apply=APPLY_SPA_CALIBRATION,
    )
    if spa_pump_apply is not None:
        context_snapshot = getattr(session, "context_snapshot", None)
        attr = context_snapshot() if callable(context_snapshot) else getattr(session, "attributes", None)
else:
    print("WARNING: SPA pump frequency fit quality gate FAILED — skipping apply.")

print(f"SPA pump frequency optimization complete.")

## 5. Readout Discrimination with SPA — Exp 57

Measure g/e readout discrimination with SPA active.

In [ ]:
spa_disc = ReadoutGEDiscrimination(session)
spa_disc_result = spa_disc.run(n_avg=SPA_DISC_N_AVG)
spa_disc_analysis = spa_disc.analyze(spa_disc_result, update_calibration=False)
spa_disc.plot(spa_disc_analysis)

spa_disc_fidelity = float(spa_disc_analysis.metrics.get("fidelity", np.nan))
print(f"Readout discrimination fidelity with SPA: {spa_disc_fidelity:.4f}")

## 6. IQ Blob with SPA — Exp 58

3-state IQ scatter plot with SPA pumping active.

In [ ]:
spa_iq_blob = IQBlob(session)
spa_iq_result = spa_iq_blob.run(n_avg=SPA_IQ_BLOB_N_AVG)
spa_iq_analysis = spa_iq_blob.analyze(spa_iq_result, update_calibration=False)
spa_iq_blob.plot(spa_iq_analysis)

print("SPA IQ blob complete.")

## 7. Save Checkpoint

In [ ]:
spa_flux_applied = "spa_flux_apply" in dir() and spa_flux_apply is not None
spa_pump_applied = "spa_pump_apply" in dir() and spa_pump_apply is not None

stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="19_spa_optimization",
    status="calibrated" if (spa_flux_applied and spa_pump_applied) else "characterized",
    summary="SPA flux and pump frequency optimization, readout discrimination, and IQ blob with SPA.",
    consumed_inputs={
        "spa_flux_n_avg": SPA_FLUX_N_AVG,
        "spa_pump_n_avg": SPA_PUMP_N_AVG,
        "spa_disc_n_avg": SPA_DISC_N_AVG,
        "spa_iq_blob_n_avg": SPA_IQ_BLOB_N_AVG,
    },
    persisted_outputs={
        "spa_flux_applied": spa_flux_applied,
        "spa_pump_applied": spa_pump_applied,
    },
    advisory_outputs={
        "spa_disc_fidelity": spa_disc_fidelity if "spa_disc_fidelity" in dir() else None,
    },
    next_stage="20_readout_leakage_benchmarking",
    notes=[
        "SPA flux and pump frequency calibrations are applied — downstream readout benefits from SPA gain.",
        "Discrimination and IQ blob are characterization-only with SPA.",
    ],
    metrics={
        "spa_disc_fidelity": spa_disc_fidelity if "spa_disc_fidelity" in dir() else None,
    },
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")